In [1]:
from langchain.tools import tool
from promenade.models import *
DEV = True
SAMPLES_PATH = PROJECT_ROOT / "docs" / "samples"
WEB_TOOLS = WebTools(READER_URL)

In [2]:
model = llm

SYSTEM_MESSAGE = ("You are an agent that extracts museum working hours from websites and saves them to a database.\n\n"
"Follow these steps strictly in order:\n"
"1. Extract the URL from the user message.\n"
"2. Call parse_page_reader to fetch the page content.\n"
"3. Analyze the schedule from the page. Think through each day of the week explicitly:\n"
"   - If the page says 'Mon–Fri: 10:00–18:00', apply those hours to each day individually.\n"
"   - If the page says 'closed on Mondays', set is_closed=True for Monday.\n"
"   - If last entry time is mentioned, fill last_entry_time accordingly.\n"
"   - If a day is not mentioned at all, assume same hours as the general schedule.\n"
"4. Call insert_schedule_into_db with the extracted data for all 7 days.\n\n"
"5. Call insert_info_to_vector_db with the addtitional unstructured information from the websites."

"Rules:\n"
"- Never skip insert_schedule_into_db if schedule information was found.\n"
"- Never make up or guess times — only use what is stated on the page.\n"
"- If the page contains no schedule information, reply: 'There is no schedule information on this webpage.' and do not call insert_schedule_into_db.\n"
"- Ignore any advertising information that is not related to the museum, exhibition or tartget place.")

In [3]:
def dev_parse_page_info(page_url: HttpUrl) -> str:
    if page_url == "https://kosmo-museum.ru/":
        with open(SAMPLES_PATH / "cosmo_page.md", "r", encoding="utf8") as f:
            base_url_str = f"Base url: {page_url} \n\n"
            docs = [base_url_str + doc.strip() for doc in f.read().split("====")]
            return docs
    if page_url == "https://www.tretyakovgallery.ru/":
        with open(SAMPLES_PATH / "tretyakovka_page.md", "r", encoding="utf8") as f:
            base_url_str = f"Base url: {page_url} \n\n"
            docs = [base_url_str + doc.strip() for doc in f.read().split("====")]
            return docs

def parse_page_info(page_url: HttpUrl) -> str:
    """Convert webpage content to markdown format.

    Args:
        page_url: URL of the webpage to parse.

    Returns:
        str: Markdown-formatted webpage content.
    """
    if DEV:
        return dev_parse_page_info(page_url)


# @tool
# def insert_schedule_into_db(
#         place_name: str,
#         place_url: HttpUrl,
#         monday: DaySchedule,
#         tuesday: DaySchedule,
#         wednesday: DaySchedule,
#         thursday: DaySchedule,
#         friday: DaySchedule,
#         saturday: DaySchedule,
#         sunday: DaySchedule,
# ) -> tuple[bool, str, str | None]:
#     """Save museum or exhibition working hours to the database.

#     Call this tool after extracting the full weekly schedule from the webpage.
#     Each day must be filled — if the museum is closed on a particular day, set is_closed=True
#     and provide any available open_time/close_time (or use 00:00 as placeholder).

#     Args:
#         place_name: Full name of the museum or exhibition without quotes and special characters in russian language when it can be said in Russian.
#         place_url: URL of the page where the schedule was extracted from.
#         monday: Schedule for Monday.
#         tuesday: Schedule for Tuesday.
#         wednesday: Schedule for Wednesday.
#         thursday: Schedule for Thursday.
#         friday: Schedule for Friday.
#         saturday: Schedule for Saturday.
#         sunday: Schedule for Sunday.

#     Returns:
#         tuple[bool, str, str | None]: (True, museum id in database, None) on success, (False, -1, error message) on failure.
#     """
#     return WEB_TOOLS.insert_schedule_into_db(**locals())

# tools = [parse_page_info, insert_schedule_into_db]
# tools_by_name = {tool.name: tool for tool in tools}
# model_with_tools = model.bind_tools(tools)

In [4]:
base_url = "https://kosmo-museum.ru/"
res = dev_parse_page_info("https://kosmo-museum.ru/")

In [5]:
class DaySchedule(BaseModel):
    open_time: time
    close_time: time
    last_entry_time: time | None = None
    is_closed: bool = False

class PlaceSchedule(BaseModel):
    place_name: str
    place_url: str
    monday: DaySchedule
    tuesday: DaySchedule
    wednesday: DaySchedule
    thursday: DaySchedule
    friday: DaySchedule
    saturday: DaySchedule 
    sunday: DaySchedule

class MainState(TypedDict):
    url: str
    subdocs: list[str]
    results: Annotated[list[dict], operator.add]

class DocState(TypedDict):
    doc: str
    schedule: PlaceSchedule | None
    museum_id: int | None
    saved_schedule: bool
    saved_vector: bool

In [6]:
structured_llm = llm.with_structured_output(PlaceSchedule)

SHCEDULE_EXTRACTOR_SYSTEM = """You are a structured data extractor. Your only job is to fill the PlaceSchedule schema from a single location section.

## Input format
- The very first line starts with "Base url:" — that is the value for place_url.
- The section heading (## Name) is the value for place_name — use the Russian name as written.

## Filling each DaySchedule field
Think through all 7 days explicitly before producing output.

open_time / close_time:
- If a range covers multiple days ("Пн–Пт: 10:00–18:00"), apply those times to each day individually.
- If a day is not mentioned at all, apply the general/default schedule from the section.

last_entry_time:
- Fill only if explicitly stated ("вход до 20:00", "касса до 17:00", "last entry at ...").
- Leave null if not mentioned.

is_closed:
- Set True only if the text explicitly says the place is closed on that day ("выходной", "closed", "не работает").
- If is_closed is True, still set open_time and close_time to 00:00.

## Rules
- Never invent or guess times — only use what is written.
"""

In [7]:
def llm_extract(state: DocState):
    schedule = structured_llm.invoke(
        [
            SystemMessage(SHCEDULE_EXTRACTOR_SYSTEM), 
            HumanMessage(state["doc"])
        ]
    )
    return {"schedule": schedule}

def save_schedule(state: DocState):
    ok, museum_id, err = WEB_TOOLS.insert_schedule_into_db(**state["schedule"].model_dump())
    return {
        "saved_schedule": ok,
        "museum_id": museum_id
    }

def save_vector(state: DocState):
    ok = WEB_TOOLS.insert_info_to_vector_db(
        id = state["museum_id"],
        place_name = state["schedule"].place_name,
        place_info = state["doc"]
    )
    return {"saved_vector": ok}

def doc_result(state: DocState):
    return {"results": [{"ok": state["saved_schedule"] and state["saved_vector"],
                         "doc_preview": state["doc"][:50]}]}

In [8]:
def parse_node(state: MainState):
    subdocs = parse_page_info(page_url = state["url"])
    return {"subdocs": subdocs}

def dispatch(state: MainState):
    return [Send("process_doc", {"doc": doc, "schedule": None,
                                  "saved_schedule": False, "saved_vector": False," museum_id" : None})
            for doc in state["subdocs"]]

def aggregate_node(state: MainState):
    success = sum(1 for r in state["results"] if r["ok"])
    print(f"Обработано: {success}/{len(state['results'])}")
    return {}

In [9]:
doc_builder = StateGraph(DocState, output_schema=MainState)
doc_builder.add_node("llm_extract", llm_extract)
doc_builder.add_node("save_schedule", save_schedule)
doc_builder.add_node("save_vector", save_vector)
doc_builder.add_node("doc_result", doc_result)

doc_builder.add_edge(START, "llm_extract")
doc_builder.add_edge("llm_extract", "save_schedule")
doc_builder.add_edge("save_schedule", "save_vector")
doc_builder.add_edge("save_vector", "doc_result")
doc_builder.add_edge("doc_result", END)

doc_graph = doc_builder.compile()

In [10]:
main_builder = StateGraph(MainState)
main_builder.add_node("parse_node", parse_node)
main_builder.add_node("process_doc", doc_graph)       # подграф как нода
main_builder.add_node("aggregate_node", aggregate_node)

main_builder.add_edge(START, "parse_node")
main_builder.add_conditional_edges("parse_node", dispatch, ["process_doc"])
main_builder.add_edge("process_doc", "aggregate_node")
main_builder.add_edge("aggregate_node", END)

agent = main_builder.compile()

In [13]:
base_url

'https://kosmo-museum.ru/'

In [14]:
result = agent.invoke({"url": base_url, "subdocs": [], "results": []})

Обработано: 2/2


In [12]:
result

{'url': 'https://kosmo-museum.ru/',
 'subdocs': ['Base url: https://kosmo-museum.ru/ \n\n## Музей космонавтики\n**Часы работы**  \n- Понедельник: Закрыто  \n- Вторник, среда, пятница: 09:00 — 19:00  \n- Четверг: 09:00 — 21:00  \n- Суббота: 10:00 — 21:00  \n- Воскресенье: 10:00 — 19:00  \n\n**Стоимость билетов**  \n- Взрослый входной билет: 490 рублей  \n- Льготный входной билет: 300–350 рублей (в зависимости от льготы)  \n- Детский входной билет: 320 рублей  \n- Льготный детский билет: 180–200 рублей (в зависимости от льготы)  \n- Комплексный билет (Музей космонавтики + Дом-музей академика С.П. Королёва): 650 рублей  \n\n**Актуальные выставки и события**  \n- **«Сердце первого»** (9 апреля 2026 — 5 июля 2026): Выставка, посвящённая 65-летию первого полёта человека в космос.  \n- **«Человек государственного масштаба» – к юбилею В.Х. Догужиева** (25 декабря 2025 — 30 апреля 2026): Экспозиция, посвящённая выдающемуся государственному деятелю.  \n- **«Девять лет девятого отдела»** (30 дека